In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración para Docker)
# ==========================================================
def iniciar_navegador():
    options = Options()
    # Esta ruta es la que configuramos en tu Dockerfile
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DE LA BÚSQUEDA 
# ==========================================================
# url del sitio
URL_OBJETIVO = "https://www.python.org/jobs/" 

# Estos selectores son específicos para la lista de empleos de Python
SELECTOR_CONTENEDOR = "li" 
SELECTOR_DATO_A     = "span.listing-job-type"         # Ejemplo: "Back end"
SELECTOR_DATO_B     = "span.listing-company-category"  # Ejemplo: "Developer"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a: {URL_OBJETIVO}...")
    driver.get(URL_OBJETIVO)
    
    # Esperamos 5 segundos para que cargue bien la lista
    time.sleep(5) 

    # Capturamos los elementos de la lista
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} líneas analizadas.")

    for item in elementos:
        try:
            # Intentamos extraer el texto de las etiquetas
            tipo_trabajo = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            categoria = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

            # Guardamos los datos en la lista
            dataset_final.append({
                "tipo_puesto": tipo_trabajo.strip(),
                "categoria": categoria.strip(),
                "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M"),
                "status": 1.0 
            })
        except:
            # Si el 'li' no tiene los datos (es publicidad o menú), lo saltamos
            continue 

    # ==========================================================
    # 4. GUARDAR Y MOSTRAR RESULTADOS
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "scraping_python_jobs.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" ¡Éxito! Archivo generado: {nombre_archivo}")
        print("\n--- Muestra de los datos extraídos ---")
        print(df.head()) 
    else:
        print(" No se capturaron datos. Revisa la URL o los selectores.")

except Exception as e:
    print(f" Error crítico: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado correctamente.")

📡 Conectando a: https://www.python.org/jobs/...
✅ Se encontraron 146 líneas analizadas.
📊 ¡Éxito! Archivo generado: scraping_python_jobs.csv

--- Muestra de los datos extraídos ---
                                         tipo_puesto             categoria  \
0                             Cloud, Systems, DevOps  Developer / Engineer   
1                   Back end, Database, Testing, Web  Developer / Engineer   
2                                           Back end  Developer / Engineer   
3  Back end, Big Data, Cloud, Database, Machine L...  Developer / Engineer   
4                         Big Data, Image Processing  Developer / Engineer   

   fecha_extraccion  status  
0  2026-04-01 02:10     1.0  
1  2026-04-01 02:10     1.0  
2  2026-04-01 02:10     1.0  
3  2026-04-01 02:10     1.0  
4  2026-04-01 02:10     1.0  
🔒 Navegador cerrado correctamente.
